# NFF-MTL on Steam — final runs and analysis (v0.3.1, clean)

This notebook only **reads** the recorded final runs and recomputes the analyses. "Run all" is safe: it never trains a
model or evaluates the test year. Every reported number comes from the run folders listed in the ledger (section 4).

The full lab record with all original cells and outputs is the archive copy
`NFF_MTL_Corrected_Colab_v0.3.1_archive.ipynb`; keep it read-only. New experiments, if any, must not be run from this
notebook (see section 8).

Times in folder names and logs are UTC (Türkiye = UTC+3).

In [1]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd

def run(*args):
    """Run a command, stream its output, raise CalledProcessError on a non-zero exit; return RUN_DIR if printed."""
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    run_dir = None
    for line in proc.stdout:
        print(line, end='')
        if line.startswith('RUN_DIR='):
            run_dir = line.strip().split('=', 1)[1]
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, args)
    return run_dir

def cli(*args):
    """Run `python -m steam_nffmtl.cli` (used here only for `compare`)."""
    return run(sys.executable, '-m', 'steam_nffmtl.cli', *args)

## 1. Setup

Any runtime works; the analysis cells use only the CPU. The package is unzipped only if it is not already present.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/steamRS')
REPO = Path('/content/steam-nffmtl-v0.3.1')
if not REPO.exists():
    run('unzip', '-q', '-o', DATA_DIR / 'steam-nffmtl-v0.3.1.zip', '-d', '/content/')
%cd {REPO}
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt')
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]')
sys.path.insert(0, str(REPO / 'src'))

Mounted at /content/drive
/content/steam-nffmtl-v0.3.1


In [3]:
RF = DATA_DIR / 'runs_final'
FINAL_CFG = DATA_DIR / 'runs_tuning' / '2026-11-09-102238-tuning-tune' / 'final_selected.yaml'
FINAL_SET = ['--set', 'train.max_epochs=10']   # every final command used this override (decided on validation data)
COMPARE = {'test-positive': RF / '2026-11-18-144556-final-compare',
           'test-interaction': RF / '2026-11-18-145125-final-compare'}

RUN_CHECKS = False      # preflight + test suite (~1 minute)
RERUN_COMPARE = False   # recompute both comparisons from the ledger (~6 minutes each, CPU only)
MARK_ORPHANS = True   # rename unfinished, unused training folders to *-INTERRUPTED (section 5)
RERUN_AUDIT = False     # re-run check_hours.py even if hours_report.json exists

assert FINAL_CFG.exists(), FINAL_CFG
assert all(d.exists() for d in COMPARE.values()), COMPARE

## 2. Optional checks

In [4]:
if RUN_CHECKS:
    cli('preflight', '--require-cuda', '--out', DATA_DIR / 'preflight')
    run(sys.executable, '-m', 'pytest', '-q', '-rs')
else:
    print('Checks skipped (RUN_CHECKS = False).')

Checks skipped (RUN_CHECKS = False).


## 3. Experiment record (what was run — do not re-run here)

| Stage | Date (UTC) | Command | Folder(s) |
| --- | --- | --- | --- |
| Prepare | 2026-11-09 | `prepare --config configs/default.yaml` | `runs_tuning/2026-11-09-093332-tuning-prepare` |
| Probe | 2026-11-09 | `train --config configs/probe.yaml --variant full --seeds 0`; `baselines --config configs/probe.yaml --split val --seed 0` | `runs_probe/2026-11-09-093942-pilot-train-full-seed0`, `runs_probe/2026-11-09-094046-pilot-baselines-val-seed0` |
| Pilot | 2026-11-09 | `train --config configs/pilot.yaml --variant full --seeds 0` | `runs_pilot/2026-11-09-094721-pilot-train-full-seed0` |
| Tuning | 2026-11-09 | `tune --config configs/default.yaml --models tagknn itemknn bpr als nffmtl --set tuning.n_trials=3 train.max_epochs=5` | `runs_tuning/2026-11-09-102238-tuning-tune` (`final_selected.yaml`) |
| 10-epoch check (validation only) | 2026-11-09 | `train --config FINAL_CFG --variant full --seeds 0 --set experiment.stage=tuning train.max_epochs=10` | `runs_final/2026-11-09-124130-tuning-train-full-seed0` |
| Final NFF-MTL: 3 variants × seeds 0–4 | 2026-11-10 → 2026-11-17 | `train --config FINAL_CFG --variant V --seeds S --set train.max_epochs=10`, then `evaluate --config FINAL_CFG --split test --run <train folder> --set train.max_epochs=10` | ledger (section 4) |
| Final baselines | 2026-11-10, 2026-11-18 | seed 0: all models; seeds 1–4: `--only Random BPR-MF ALS` | ledger |
| EASE tuning | 2026-11-18 | `tune --config configs/default.yaml --models ease --set baselines.ease.enabled=true tuning.n_trials=3` → `l2 = 100` | `runs_tuning/2026-11-18-103721-tuning-tune` |
| EASE test | 2026-11-18 | `baselines --config FINAL_CFG --split test --seed 0 --only EASE --set baselines.ease.enabled=true baselines.ease.l2=100.0 baselines.ease.device=cpu train.max_epochs=10` | `runs_final/2026-11-18-142502-final-baselines-test-seed0` |
| Comparisons | 2026-11-18 | `compare --config FINAL_CFG --reference NFF-MTL --split test-positive` (and `test-interaction`) | `runs_final/2026-11-18-144556-final-compare`, `runs_final/2026-11-18-145125-final-compare` |

**Deviations and incidents to report**

- Hardware: Colab L4 (not A100); Python 3.13, torch 2.14, numpy 2.4, pandas 2.2 (exact versions in each run's `env.json`). The test suite passed 63/63 on this runtime.
- Tuning used 3 candidates per model (equal budget). NFF-MTL trials were capped at 5 epochs; the cap was raised to 10 after one extra validation-only run, so NFF-MTL received one validation run more than the other models.
- EASE was added after the other models' test results were seen. It was tuned on validation only, with the same 3-trial budget. On the L4 the float64 inversion did not finish within an hour, so the test run used the CPU inversion (same computation).
- Interrupting a notebook cell does not stop the CLI process: the seed-2 training of 2026-11-12 finished in the background and is the one used. Unused attempts are renamed `*-INTERRUPTED`: EASE `2026-11-18-112503`, `2026-11-18-122839`; full seed 2 `2026-11-12-144140`. The partial full-seed-2 run `2026-11-14-151106` (accidental re-run, stopped after 3 epochs) is unused and is marked in section 5.

## 4. Final run ledger

The single source of truth: 15 NFF-MTL evaluation folders (3 variants × 5 seeds) and 6 baseline folders
(seed 0 with all models, seeds 1–4 with Random/BPR-MF/ALS, and EASE).

In [5]:
LEDGER = {
    'baselines': ['2026-11-10-114002-final-baselines-test-seed0', '2026-11-18-085122-final-baselines-test-seed1',
                  '2026-11-18-091445-final-baselines-test-seed2', '2026-11-18-093726-final-baselines-test-seed3',
                  '2026-11-18-100018-final-baselines-test-seed4', '2026-11-18-142502-final-baselines-test-seed0'],
    'full': ['2026-11-10-112426-final-evaluate-test-full-seed0', '2026-11-12-110556-final-evaluate-test-full-seed1',
             '2026-11-12-144748-final-evaluate-test-full-seed2', '2026-11-13-121946-final-evaluate-test-full-seed3',
             '2026-11-13-135138-final-evaluate-test-full-seed4'],
    'no_content': ['2026-11-13-190729-final-evaluate-test-no_content-seed0', '2026-11-15-103653-final-evaluate-test-no_content-seed1',
                   '2026-11-15-131526-final-evaluate-test-no_content-seed2', '2026-11-16-094010-final-evaluate-test-no_content-seed3',
                   '2026-11-16-124644-final-evaluate-test-no_content-seed4'],
    'single_task': ['2026-11-17-110731-final-evaluate-test-single_task-seed0', '2026-11-17-191017-final-evaluate-test-single_task-seed1',
                    '2026-11-17-202246-final-evaluate-test-single_task-seed2', '2026-11-17-215014-final-evaluate-test-single_task-seed3',
                    '2026-11-17-231725-final-evaluate-test-single_task-seed4'],
}
result_dirs = [RF / name for names in LEDGER.values() for name in names]
missing = [p.name for p in result_dirs if not list((p / 'results').glob('*__test-positive.json'))]
assert not missing, f'Missing or incomplete result folders: {missing}'
rows = []
for p in result_dirs:
    m = json.loads((p / 'manifest.json').read_text())
    models = sorted({json.loads(j.read_text())['model'] for j in (p / 'results').glob('*__test-positive.json')})
    rows.append({'folder': p.name, 'command': m['command'], 'variant': m.get('variant', '-'), 'seed': m.get('seed'),
                 'models': ', '.join(models), 'train_folder': Path(m['checkpoint']).parent.name if 'checkpoint' in m else ''})
ledger = pd.DataFrame(rows)
print(len(ledger), 'result folders (expected 21)')
display(ledger)

21 result folders (expected 21)


,folder,command,variant,seed,models,train_folder
0,2026-11-10-114002-final-baselines-test-seed0,baselines,-,0,"ALS, BPR-MF, ItemKNN-CF, Popularity, Random, T...",
1,2026-11-18-085122-final-baselines-test-seed1,baselines,-,1,"ALS, BPR-MF, Random",
2,2026-11-18-091445-final-baselines-test-seed2,baselines,-,2,"ALS, BPR-MF, Random",
3,2026-11-18-093726-final-baselines-test-seed3,baselines,-,3,"ALS, BPR-MF, Random",
4,2026-11-18-100018-final-baselines-test-seed4,baselines,-,4,"ALS, BPR-MF, Random",
5,2026-11-18-142502-final-baselines-test-seed0,baselines,-,0,EASE,
6,2026-11-10-112426-final-evaluate-test-full-seed0,evaluate,full,0,NFF-MTL,2026-11-10-095955-final-train-full-seed0
7,2026-11-12-110556-final-evaluate-test-full-seed1,evaluate,full,1,NFF-MTL,2026-11-10-134358-final-train-full-seed1
8,2026-11-12-144748-final-evaluate-test-full-seed2,evaluate,full,2,NFF-MTL,2026-11-12-132154-final-train-full-seed2
9,2026-11-13-121946-final-evaluate-test-full-seed3,evaluate,full,3,NFF-MTL,2026-11-13-094655-final-train-full-seed3


## 5. Drive hygiene

Lists training folders that no evaluation uses. Unfinished ones are renamed `*-INTERRUPTED` only when
`MARK_ORPHANS = True`, and only while no CLI process is running.

In [6]:
used = set(ledger['train_folder']) - {''}
orphans = [p for p in sorted(RF.glob('*-final-train-*')) if 'INTERRUPTED' not in p.name and p.name not in used]
for p in orphans:
    print('unused training folder:', p.name, '| finished:', (p / 'training.json').exists())
print('already marked:', [p.name for p in sorted(RF.glob('*INTERRUPTED*'))])
if MARK_ORPHANS:
    busy = subprocess.run(['pgrep', '-f', 'steam_nffmtl.cli'], capture_output=True, text=True).stdout.strip()
    assert not busy, 'A steam_nffmtl CLI process is running; wait until it finishes.'
    for p in orphans:
        if not (p / 'training.json').exists():
            p.rename(p.with_name(p.name + '-INTERRUPTED'))
            print('marked:', p.name)

unused training folder: 2026-11-14-151106-final-train-full-seed2 | finished: False
already marked: ['2026-11-12-144140-final-train-full-seed2-INTERRUPTED', '2026-11-18-112503-final-baselines-test-seed0-INTERRUPTED', '2026-11-18-122839-final-baselines-test-seed0-INTERRUPTED']
marked: 2026-11-14-151106-final-train-full-seed2


## 6. Comparison tables and statistics

In [7]:
if RERUN_COMPARE:
    for split in COMPARE:
        COMPARE[split] = Path(cli('compare', '--config', FINAL_CFG, '--reference', 'NFF-MTL', '--split', split,
                                  '--results', *result_dirs, *FINAL_SET))
COLS = ['runs', 'ndcg@10_mean', 'ndcg@20_mean', 'ndcg@20_sd', 'recall@10_mean', 'recall@20_mean', 'recall@20_sd',
        'precision@10_mean', 'hr@20_mean', 'coverage@20_mean', 'long_tail_share@20_mean', 'arp@20_mean',
        'auc_review_label_mean']
tables = {s: pd.read_csv(d / 'comparison_table.csv', index_col=0).sort_values('ndcg@20_mean', ascending=False)
          for s, d in COMPARE.items()}
for s, t in tables.items():
    print(s, '-', COMPARE[s].name)
    display(t[[c for c in COLS if c in t.columns]].round(4))

test-positive - 2026-11-18-144556-final-compare


,runs,ndcg@10_mean,ndcg@20_mean,ndcg@20_sd,recall@10_mean,recall@20_mean,recall@20_sd,precision@10_mean,hr@20_mean,coverage@20_mean,long_tail_share@20_mean,arp@20_mean,auc_review_label_mean
model,,,,,,,,,,,,,
EASE,1,0.0472,0.0615,NaN,0.0769,0.1260,NaN,0.0178,0.2329,0.3442,0.0073,21973.3582,0.5692
ALS,5,0.0411,0.0528,0.0002,0.0654,0.1056,0.0004,0.0151,0.2002,0.0368,0.0000,19070.3291,0.5565
ItemKNN-CF,1,0.0376,0.0487,NaN,0.0594,0.0977,NaN,0.0145,0.1881,0.4798,0.0255,25293.4240,0.5784
NFF-MTL[single_task],5,0.0266,0.0363,0.0010,0.0440,0.0771,0.0021,0.0105,0.1548,0.2892,0.0022,24177.9649,0.6451
NFF-MTL,5,0.0234,0.0332,0.0011,0.0402,0.0734,0.0017,0.0095,0.1481,0.1562,0.0007,27595.9707,0.6462
NFF-MTL[no_content],5,0.0211,0.0305,0.0016,0.0368,0.0690,0.0035,0.0089,0.1407,0.1166,0.0005,28420.2457,0.6463
BPR-MF,5,0.0231,0.0300,0.0003,0.0371,0.0606,0.0005,0.0086,0.1196,0.5008,0.2125,4952.1894,0.5298
Popularity,1,0.0187,0.0280,NaN,0.0291,0.0614,NaN,0.0072,0.1262,0.0030,0.0000,44831.9738,0.5602
TagKNN,1,0.0052,0.0066,NaN,0.0076,0.0123,NaN,0.0023,0.0330,0.9149,0.3377,2491.7994,0.5036


test-interaction - 2026-11-18-145125-final-compare


,runs,ndcg@10_mean,ndcg@20_mean,ndcg@20_sd,recall@10_mean,recall@20_mean,recall@20_sd,precision@10_mean,hr@20_mean,coverage@20_mean,long_tail_share@20_mean,arp@20_mean,auc_review_label_mean
model,,,,,,,,,,,,,
EASE,1,0.0455,0.0595,NaN,0.0730,0.1204,NaN,0.0179,0.2338,0.3565,0.0075,21837.3437,0.5692
ALS,5,0.0400,0.0514,0.0001,0.0627,0.1015,0.0004,0.0154,0.2025,0.0371,0.0000,19048.0395,0.5565
ItemKNN-CF,1,0.0361,0.0470,NaN,0.0562,0.0930,NaN,0.0145,0.1882,0.5163,0.0265,25085.8274,0.5784
NFF-MTL[single_task],5,0.0244,0.0334,0.0009,0.0397,0.0698,0.0019,0.0101,0.1487,0.3013,0.0022,23842.1392,0.6451
NFF-MTL,5,0.0216,0.0306,0.0010,0.0362,0.0663,0.0013,0.0092,0.1424,0.1648,0.0007,27331.2262,0.6462
BPR-MF,5,0.0228,0.0295,0.0003,0.0360,0.0589,0.0005,0.0088,0.1224,0.5057,0.2156,4941.0150,0.5298
Popularity,1,0.0189,0.0282,NaN,0.0285,0.0606,NaN,0.0076,0.1310,0.0030,0.0000,44851.0904,0.5602
NFF-MTL[no_content],5,0.0194,0.0281,0.0016,0.0331,0.0624,0.0034,0.0085,0.1356,0.1229,0.0005,28208.4667,0.6463
TagKNN,1,0.0051,0.0065,NaN,0.0072,0.0118,NaN,0.0024,0.0338,0.9214,0.3395,2511.7968,0.5036


In [8]:
for s, d in COMPARE.items():
    st = json.loads((d / 'comparison_stats.json').read_text())
    for metric in ['ndcg@20', 'recall@20']:
        holm = st['holm_seed_level'].get(metric, {})
        print(f'\n{s} | {metric} | seed-level paired t-test, Holm over {len(holm)} comparisons (difference = NFF-MTL - other)')
        for key, v in st['seed_level'].items():
            if key.endswith('| ' + metric):
                other = key.split(' vs ')[1].split(' |')[0]
                lo, hi = v['ci95_seed_t']
                print(f'  {other:22s} diff {v["mean_diff"]:+.5f}  95% CI [{lo:+.5f}, {hi:+.5f}]  '
                      f'd_z {v["d_z"]:+.2f}  Holm p {holm.get(other, float("nan")):.4f}')


test-positive | ndcg@20 | seed-level paired t-test, Holm over 9 comparisons (difference = NFF-MTL - other)
  ALS                    diff -0.01956  95% CI [-0.02098, -0.01815]  d_z -17.13  Holm p 0.0000
  BPR-MF                 diff +0.00322  95% CI [+0.00197, +0.00447]  d_z +3.20  Holm p 0.0060
  EASE                   diff -0.02834  95% CI [-0.02970, -0.02698]  d_z -25.83  Holm p 0.0000
  ItemKNN-CF             diff -0.01556  95% CI [-0.01692, -0.01420]  d_z -14.18  Holm p 0.0000
  NFF-MTL[no_content]    diff +0.00267  95% CI [+0.00030, +0.00503]  d_z +1.40  Holm p 0.0413
  NFF-MTL[single_task]   diff -0.00314  95% CI [-0.00549, -0.00079]  d_z -1.66  Holm p 0.0413
  Popularity             diff +0.00522  95% CI [+0.00386, +0.00658]  d_z +4.76  Holm p 0.0018
  Random                 diff +0.03281  95% CI [+0.03144, +0.03418]  d_z +29.80  Holm p 0.0000
  TagKNN                 diff +0.02656  95% CI [+0.02520, +0.02792]  d_z +24.21  Holm p 0.0000

test-positive | recall@20 | seed-level p

In [9]:
per_seed = []
for p in result_dirs:
    for j in (p / 'results').glob('NFF-MTL*__test-*.json'):
        m = json.loads(j.read_text())
        per_seed.append({'model': m['model'], 'split': m['split'], 'seed': m['seed'],
                         'ndcg@20': m['summary']['ndcg@20'], 'recall@20': m['summary']['recall@20']})
per_seed = pd.DataFrame(per_seed)
display(per_seed.pivot_table(index=['split', 'seed'], columns='model', values='ndcg@20').round(5))

model                  NFF-MTL  NFF-MTL[no_content]  NFF-MTL[single_task]
split            seed                                                    
test-interaction 0     0.03000              0.02678               0.03285
                 1     0.03042              0.02901               0.03371
                 2     0.03221              0.02790               0.03211
                 3     0.02986              0.02645               0.03437
                 4     0.03028              0.03031               0.03384
test-positive    0     0.03261              0.02918               0.03548
                 1     0.03284              0.03140               0.03688
                 2     0.03513              0.03036               0.03516
                 3     0.03252              0.02887               0.03746
                 4     0.03283              0.03280               0.03666

## 7. Playtime head and dataset audit

In [10]:
pt = []
for p in result_dirs:
    f = p / 'playtime.json'
    if f.exists():
        m = json.loads((p / 'manifest.json').read_text())
        for name, r in json.loads(f.read_text())['metrics'].items():
            pt.append({'variant': m.get('variant'), 'seed': m.get('seed'), 'predictor': name, 'mae_log': r['mae_log'],
                       'rmse_log': r['rmse_log'], 'spearman_log': r.get('spearman_rho_log')})
pt = pd.DataFrame(pt)
display(pt.groupby(['variant', 'predictor'])[['mae_log', 'rmse_log', 'spearman_log']].agg(['mean', 'std']).round(4))

mae_log         rmse_log         spearman_log  \
                                  mean     std     mean     std         mean   
variant    predictor                                                           
full       GameMedian           0.9101  0.0000   1.2398  0.0000       0.6141   
           GlobalMean           1.2425  0.0000   1.5195  0.0000          NaN   
           NFF-MTL              0.8914  0.0052   1.1969  0.0037       0.6275   
           UserGameMean         0.9116  0.0000   1.2160  0.0000       0.6406   
no_content GameMedian           0.9101  0.0000   1.2398  0.0000       0.6141   
           GlobalMean           1.2425  0.0000   1.5195  0.0000          NaN   
           NFF-MTL[no_content]  0.8845  0.0098   1.1918  0.0155       0.6306   
           UserGameMean         0.9116  0.0000   1.2160  0.0000       0.6406   

                                        
                                   std  
variant    predictor                    
full       GameMedian           0.0000  
           GlobalMean              NaN  
           NFF-MTL              0.0036  
           UserGameMean         0.0000  
no_content GameMedian           0.0000  
           GlobalMean              NaN  
           NFF-MTL[no_content]  0.0078  
           UserGameMean         0.0000

In [11]:
CHECK, REPORT = DATA_DIR / 'check_hours.py', DATA_DIR / 'hours_report.json'
if CHECK.exists() and (RERUN_AUDIT or not REPORT.exists()):
    args = ['--recs', DATA_DIR / 'recommendations.csv', '--out', REPORT]
    if (DATA_DIR / 'games.csv').exists():
        args += ['--games', DATA_DIR / 'games.csv', '--meta', DATA_DIR / 'games_metadata.json']
    run(sys.executable, CHECK, *args)
if REPORT.exists():
    rep = json.loads(REPORT.read_text())
    print(json.dumps({'hours_profile': rep['hours_profile'], 'release_check': rep['release_check']}, indent=1))
else:
    print('Upload check_hours.py to', DATA_DIR, 'to run the audit.')

{
 "hours_profile": {
  "rows": 41154794,
  "min": 0.0,
  "max": 1000.0,
  "q50_q90_q99_q999": [
   27.3,
   304.0,
   864.5,
   985.2
  ],
  "rows_ge_1000": 26,
  "rows_eq_0": 176329,
  "share_lt_1h": 0.052526663114873084,
  "consistent_with_comma_truncation": false
 },
 "heavy_games": {
  "share_lt_1h_all_games": 0.052526663114873084,
  "games": {
   "Dota 2": {
    "reviews": 216914,
    "share_lt_1h": 0.01717731451174198,
    "share_ge_500h": 0.417621730270983,
    "max": 999.9000244140625
   },
   "Counter-Strike: GO": {
    "reviews": 219737,
    "share_lt_1h": 0.001337963110445669,
    "share_ge_500h": 0.39681528372554464,
    "max": 999.9000244140625
   },
   "Team Fortress 2": {
    "reviews": 319492,
    "share_lt_1h": 0.020050580296220248,
    "share_ge_500h": 0.2725169957307225,
    "max": 999.9000244140625
   },
   "PUBG": {
    "reviews": 97733,
    "share_lt_1h": 0.008400437927823764,
    "share_ge_500h": 0.2397552515526997,
    "max": 999.7999877929688
   }
  }
 }
}
{
 

In [12]:
OUT = DATA_DIR / 'paper_tables'
OUT.mkdir(exist_ok=True)
for s, t in tables.items():
    t.to_csv(OUT / f'comparison_{s}.csv')
per_seed.to_csv(OUT / 'nffmtl_per_seed.csv', index=False)
pt.to_csv(OUT / 'playtime_per_seed.csv', index=False)
print('written:', sorted(p.name for p in OUT.iterdir()))

written: ['comparison_test-interaction.csv', 'comparison_test-positive.csv', 'nffmtl_per_seed.csv', 'playtime_per_seed.csv']


## 8. Notes for any further experiment

- Run new experiments in a separate notebook and a **new** runs folder; never add them to this ledger without
  recording why. Test-year evaluations are limited by the registry next to the data cache.
- A notebook interrupt does not stop the CLI subprocess. Check with `ps -eo pid,etime,cmd | grep steam_nffmtl` in the
  terminal before restarting anything; stop a process with `kill <PID>`.
- On an L4, use `baselines.ease.device=cpu` for EASE.
- The OpenBLAS and implicit-GPU warnings in the logs affect speed only.